# 📊 Training ADITS 5-Class Command Classifier: Baseline LightGBM 93.33% (Sebelum Tuning)
Notebook ini merupakan **Model Baseline LightGBM (Sebelum Regularisasi/Pruning)** pada sistem **ADITS (Audio-Driven IoT Task Scheduler)** untuk mengenali 5 perintah suara verbal (`MULAI`, `STOP`, `LANJUT`, `BERDIRI`, `DUDUK`) dengan pembagian data **70% Data Latih (Training) : 30% Data Uji (Testing)**.

**Catatan Evaluasi (Bahan Laporan Dosen):**
- Pada model *Baseline* ini, ke-13 parameter ditulis lengkap dengan nilai bawaan sebelum *pruning* (`num_leaves = 20`, `colsample_bytree = 0.8`, `subsample = 1.0`, `subsample_freq = 0`), menghasilkan **Akurasi Data Uji sebesar 93.33%**.
- Bandingkan baris per baris dengan notebook utama (`Training_Command_Classifier_5Kelas.ipynb`) yang menerapkan *Leaf Pruning* (`num_leaves = 12`), *Feature Subsampling* (`colsample_bytree = 0.5`), dan *Row Subsampling* (`subsample = 0.85`), di mana akurasi generalisasi pada data uji meningkat menjadi **96.00%**.

### 1. Install & Import Library

In [ ]:
!pip install lightgbm -q

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import lightgbm as lgb

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("Library LightGBM & Scikit-Learn siap digunakan!")

### 2. Upload Dataset Excel
Jalankan cell di bawah untuk meng-upload file `dataset_perintah_suara_5kelas_realistis.xlsx` dari komputermu.

In [ ]:
filename = 'dataset_perintah_suara_5kelas_realistis.xlsx'
if not os.path.exists(filename):
    print("Silakan upload file 'dataset_perintah_suara_5kelas_realistis.xlsx':")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
else:
    print(f"File {filename} sudah ada di sesi Colab!")

df = pd.read_excel(filename)
print(f"\nFile Dataset : {filename}")
print(f"Total Data   : {len(df)} sampel")
print("Distribusi Kelas:")
print(df['label'].value_counts())

### 3. Pemisahan Fitur Akustik & Split Data (70% Train : 30% Test)

In [ ]:
non_feature_cols = ['label', 'file_wav', 'rms_raw', 'kondisi_audio']
available_feats = [c for c in df.columns if c not in non_feature_cols]

X = np.nan_to_num(df[available_feats].values, nan=0.0)
le = LabelEncoder()
y = le.fit_transform(df['label'].values)
class_names = list(le.classes_)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Jumlah Fitur Akustik : {len(available_feats)} fitur")
print(f"Kelas Target         : {class_names}")
print(f"Data Latih (70%)     : {len(X_train)} sampel")
print(f"Data Uji   (30%)     : {len(X_test)} sampel")

### 4. Training Model Baseline LightGBM (Sebelum Tuning — 93.33%)

In [ ]:
lgbm_baseline = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=20,
    min_child_samples=15,
    colsample_bytree=0.8,
    subsample=1.0,
    subsample_freq=0,
    reg_alpha=0.1,
    reg_lambda=1.0,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

lgbm_baseline.fit(X_train, y_train)

y_pred_train = lgbm_baseline.predict(X_train)
y_pred_test  = lgbm_baseline.predict(X_test)

acc_train = accuracy_score(y_train, y_pred_train) * 100.0
acc_test  = accuracy_score(y_test, y_pred_test) * 100.0

print("=== HASIL TRAINING & EVALUASI BASELINE LIGHTGBM (SPLIT 70:30) ===")
print(f"Akurasi Data Latih (70% - {len(X_train)} sampel) : {acc_train:.2f} %")
print(f"Akurasi Data Uji   (30% - {len(X_test)} sampel) : {acc_test:.2f} %")

### 5. Detail Evaluasi Metrik Klasifikasi Baseline (Data Uji 30%)

In [ ]:
prec = precision_score(y_test, y_pred_test, average='macro', zero_division=0) * 100.0
rec  = recall_score(y_test, y_pred_test, average='macro', zero_division=0) * 100.0
f1   = f1_score(y_test, y_pred_test, average='macro', zero_division=0) * 100.0

print(f"Akurasi   : {acc_test:.2f} %")
print(f"Presisi   : {prec:.2f} %")
print(f"Recall    : {rec:.2f} %")
print(f"F1-Score  : {f1:.2f} %\n")
print("=== CLASSIFICATION REPORT BASELINE LIGHTGBM (DATA UJI 30%) ===")
print(classification_report(y_test, y_pred_test, target_names=class_names, digits=4, zero_division=0))

### 6. Visualisasi Confusion Matrix Baseline LightGBM (5x5)

In [ ]:
cm = confusion_matrix(y_test, y_pred_test)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt="d", cmap="Oranges",
    xticklabels=class_names, yticklabels=class_names,
    linewidths=0.5, linecolor="gray"
)
plt.xlabel("Prediksi Baseline LightGBM", fontsize=12, labelpad=10)
plt.ylabel("Label Aktual", fontsize=12, labelpad=10)
plt.title(
    f"Confusion Matrix Baseline LightGBM (Sebelum Tuning)\n"
    f"Akurasi: {acc_test:.2f}% | Presisi: {prec:.2f}% | Recall: {rec:.2f}% | F1: {f1:.2f}%",
    fontsize=11.5, pad=15
)
plt.tight_layout()
plt.savefig("confusion_matrix_baseline_93pct.png", dpi=300)
plt.show()